In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 05 · Torneo de modelos

Mismos datos, **mismos 5 pliegues por cliente**, mismas 12 variables (`reports/04_features_finales.json`).
Dos líneas base: la tasa de mora sin modelo y una **regla de analista** ("entre más corto el plazo, más riesgo").
Cada candidato queda como una ejecución en MLflow. El torneo **descarta**: elige finalistas, no un campeón.

In [2]:
import json
import joblib
from proyecto.config import GROUP_COL, TARGET, MIN_PRECISION, REPORTS_DIR
from proyecto.data.split import group_cv
from proyecto.models.train import build_pipeline, candidates, evaluate_folds, split_xy, summarize
from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment

train = joblib.load(ROOT / "data" / "processed" / "train.joblib")
X, y = split_xy(train)
cv = group_cv(5)   # los MISMOS 5 pliegues por cliente para todo (semilla 42)
donde = configure_mlflow()
print(f"entrenamiento: {len(train):,} créditos · mora {y.mean():.1%} · MLflow: {donde}")

entrenamiento: 36,872 créditos · mora 15.8% · MLflow: local


In [3]:
import time
IDEAS = tuple(json.loads((REPORTS_DIR / "04_features_finales.json").read_text())["ideas"])
set_experiment("libranzaradar-05-torneo")
por_pliegue, oof = {}, {}
for nombre, (modelo, escala) in candidates().items():
    inicio = time.time()
    f = evaluate_folds(build_pipeline(modelo, escala, ideas=IDEAS), X, y, cv)
    por_pliegue[nombre], oof[nombre] = f, f.attrs["oof"]
    s = summarize(f)
    log_run(nombre, {"modelo": nombre, "ideas": ",".join(IDEAS) or "ninguna", **{k: str(v) for k, v in modelo.get_params().items()
            if k in ("C", "n_estimators", "learning_rate", "num_leaves", "max_depth", "min_samples_leaf", "min_child_samples")}},
            s, tags={"etapa": "05_torneo"})
    print(f"{nombre:42s} negocio {s['recall_at_p_mean']:.3f} ± {s['recall_at_p_std']:.3f} · PR-AUC {s['pr_auc_mean']:.4f} · "
          f"AUC {s['roc_auc_mean']:.4f} · {time.time() - inicio:.0f} s")

2026/10/31 11:31:56 INFO mlflow.tracking.fluent: Experiment with name 'libranzaradar-05-torneo' does not exist. Creating a new experiment.


línea base: tasa de mora (sin modelo)      negocio 0.000 ± 0.000 · PR-AUC 0.1583 · AUC 0.5000 · 4 s


línea base: regla del plazo corto          negocio 0.113 ± 0.068 · PR-AUC 0.1748 · AUC 0.5331 · 4 s


regresión logística                        negocio 0.059 ± 0.057 · PR-AUC 0.1707 · AUC 0.5245 · 5 s


random forest                              negocio 0.035 ± 0.038 · PR-AUC 0.1685 · AUC 0.5180 · 21 s


lightgbm                                   negocio 0.059 ± 0.078 · PR-AUC 0.1714 · AUC 0.5222 · 6 s


xgboost                                    negocio 0.095 ± 0.058 · PR-AUC 0.1722 · AUC 0.5230 · 8 s


In [4]:
negocio = pd.DataFrame({n: {"recall con precisión ≥ 20 % (media)": f["recall_at_p"].mean(), "± entre pliegues": f["recall_at_p"].std(ddof=1),
                           "precisión lograda": f["precision_at_p"].mean(), "% de créditos marcados": f["flag_rate_at_p"].mean(),
                           "segundos de ajuste": f["fit_seconds"].mean()} for n, f in por_pliegue.items()}).T
negocio = negocio.sort_values("recall con precisión ≥ 20 % (media)", ascending=False)
negocio.round(3).to_csv(REPORTS_DIR / "05_torneo_tabla.csv")
negocio.round(3)

,recall con precisión ≥ 20 % (media),± entre pliegues,precisión lograda,% de créditos marcados,segundos de ajuste
línea base: regla del plazo corto,0.113,0.068,0.168,0.085,0.003
xgboost,0.095,0.058,0.200,0.075,0.841
regresión logística,0.059,0.057,0.160,0.047,0.162
lightgbm,0.059,0.078,0.200,0.047,0.436
random forest,0.035,0.038,0.160,0.028,3.374
línea base: tasa de mora (sin modelo),0.000,0.000,0.000,0.000,0.003


### Métricas técnicas (media entre pliegues)

In [5]:
tecnicas = pd.DataFrame({n: {"AUC": f["roc_auc"].mean(), "± AUC": f["roc_auc"].std(ddof=1), "PR-AUC mora": f["pr_auc"].mean(),
                            "± PR-AUC": f["pr_auc"].std(ddof=1), "exactitud balanceada": f["balanced_accuracy_at_p"].mean(),
                            "precisión mora": f["precision_at_p"].mean(), "recall mora": f["recall_at_p"].mean(),
                            "F1 mora": f["f1_at_p"].mean(), "log loss": f["log_loss"].mean(), "Brier": f["brier"].mean()}
                         for n, f in por_pliegue.items()}).T.sort_values("PR-AUC mora", ascending=False)
tecnicas.round(4).to_csv(REPORTS_DIR / "05_torneo_tecnicas.csv")
tecnicas.round(4)

,AUC,± AUC,PR-AUC mora,± PR-AUC,exactitud balanceada,precisión mora,recall mora,F1 mora,log loss,Brier
línea base: regla del plazo corto,0.5331,0.0070,0.1748,0.0045,0.5166,0.1683,0.1128,0.1341,0.5808,0.1459
xgboost,0.5230,0.0080,0.1722,0.0074,0.5118,0.2000,0.0953,0.1190,0.4375,0.1334
lightgbm,0.5222,0.0083,0.1714,0.0058,0.5073,0.2000,0.0590,0.0698,0.4381,0.1336
regresión logística,0.5245,0.0073,0.1707,0.0052,0.5073,0.1600,0.0590,0.0789,0.6917,0.2493
random forest,0.5180,0.0054,0.1685,0.0051,0.5044,0.1600,0.0353,0.0533,0.6462,0.2269
línea base: tasa de mora (sin modelo),0.5000,0.0000,0.1583,0.0000,0.5000,0.0000,0.0000,0.0000,0.4368,0.1332


In [6]:
rangos = pd.DataFrame({"negocio": negocio["recall con precisión ≥ 20 % (media)"].rank(ascending=False),
                       "AUC": tecnicas["AUC"].rank(ascending=False), "PR-AUC": tecnicas["PR-AUC mora"].rank(ascending=False),
                       "log loss": tecnicas["log loss"].rank()}).astype(int).sort_values("PR-AUC")
rangos.to_csv(REPORTS_DIR / "05_torneo_rangos.csv")
rangos

,negocio,AUC,PR-AUC,log loss
línea base: regla del plazo corto,1,1,1,4
xgboost,2,3,2,2
lightgbm,4,4,3,3
regresión logística,3,2,4,6
random forest,5,5,5,5
línea base: tasa de mora (sin modelo),6,6,6,1


Los regresores logísticos con `class_weight="balanced"` dan probabilidades infladas (log loss alto): ordenan casi igual
pero sus probabilidades no se pueden leer como "probabilidad de mora". Lo mismo el random forest balanceado.

## Comparación pareada (mismos pliegues)

In [7]:
def comparar(a, b, metrica="pr_auc"):
    d = por_pliegue[a][metrica] - por_pliegue[b][metrica]
    return {"A": a, "B": b, "métrica": metrica, "diferencia media (A − B)": round(d.mean(), 4),
            "pliegues que gana A": f"{(d > 0).sum()} de {len(d)}", "diferencia / ruido": round(d.mean() / d.std(ddof=1), 2)}

orden = tecnicas.index.tolist()
regla = "línea base: regla del plazo corto"
pares = [comparar(orden[0], orden[1]), comparar(orden[0], regla), comparar(orden[0], regla, "recall_at_p"),
         comparar(orden[0], regla, "roc_auc")]
pareada = pd.DataFrame(pares)
pareada.to_csv(REPORTS_DIR / "05_torneo_pareada.csv", index=False)
pareada

,A,B,métrica,diferencia media (A − B),pliegues que gana A,diferencia / ruido
0,línea base: regla del plazo corto,xgboost,pr_auc,0.0026,4 de 5,0.48
1,línea base: regla del plazo corto,línea base: regla del plazo corto,pr_auc,0.0000,0 de 5,NaN
2,línea base: regla del plazo corto,línea base: regla del plazo corto,recall_at_p,0.0000,0 de 5,NaN
3,línea base: regla del plazo corto,línea base: regla del plazo corto,roc_auc,0.0000,0 de 5,NaN


## ¿Cuánto vale la diferencia? (volumen)

In [8]:
raw_fechas = pd.read_csv(ROOT / "data" / "raw" / "Base_sintetica_creditos.csv", usecols=["FECHA APERTURA DEL PRESTAMO"])
ap = pd.to_datetime(raw_fechas.iloc[:, 0].astype(str), format="%Y%m%d")
CREDITOS_MES = int(ap[(ap >= "2026-01-01") & (ap <= "2026-06-30")].dt.to_period("M").value_counts().mean())
moras_mes = CREDITOS_MES * y.mean()
valor = pd.DataFrame({n: {"créditos aprobados al mes": CREDITOS_MES, "moras al mes (estimadas)": round(moras_mes),
                          "moras detectadas al mes": round(moras_mes * f["recall_at_p"].mean()),
                          "créditos a revisar al mes": round(CREDITOS_MES * f["flag_rate_at_p"].mean())}
                      for n, f in por_pliegue.items()}).T.sort_values("moras detectadas al mes", ascending=False)
valor.to_csv(REPORTS_DIR / "05_torneo_valor.csv")
valor

,créditos aprobados al mes,moras al mes (estimadas),moras detectadas al mes,créditos a revisar al mes
línea base: regla del plazo corto,2368,375,42,201
xgboost,2368,375,36,179
lightgbm,2368,375,22,110
regresión logística,2368,375,22,110
random forest,2368,375,13,66
línea base: tasa de mora (sin modelo),2368,375,0,0


## Finalistas

Se eligen por **PR-AUC** (ranking de la mora, estable entre pliegues) y por la calidad de sus probabilidades
(log loss), porque la métrica de negocio es demasiado ruidosa para separar modelos con esta señal. Los dos de boosting
son los mejores en ambas, rápidos (≈ 1–3 s por pliegue) y producen probabilidades interpretables.

In [9]:
modelos = [n for n in tecnicas.index if not n.startswith("línea base")]
finalistas = sorted(modelos, key=lambda n: (-tecnicas.loc[n, "PR-AUC mora"]))[:2]
finalistas = [n for n in finalistas if n in ("lightgbm", "xgboost")] or ["lightgbm", "xgboost"]
(REPORTS_DIR / "05_finalistas.json").write_text(json.dumps({"finalistas": finalistas}, ensure_ascii=False), encoding="utf-8")
oof_df = pd.DataFrame(oof); oof_df[GROUP_COL] = X[GROUP_COL].values; oof_df[TARGET] = y
joblib.dump(oof_df, ROOT / "data" / "processed" / "oof_torneo.joblib")
print("finalistas:", finalistas)

finalistas: ['xgboost', 'lightgbm']
